# OptiVision Easy Mode: Validation Scoring & Decision Suite (Parts 1–3)

Runs `scripts/easy_validation/run_all.py` on Kaggle (30 GB RAM + optional T4 GPU) across all validation batches, starting from the saved Step 1–6 checkpoint.

### Instructions:
1. **Settings** (Right panel): **Accelerator**: `GPU T4 x2` (recommended for fast MaxSim scoring) or `None` (CPU). **Internet**: `On`.
2. **Attach Notebook Outputs** (Right panel -> **+ Add Input** -> **Notebook Output** / **Your Work**):
   Select your batch runs (`e1a`, `e1b`, `e2a`, `e2b`, `e3a`, `e3b`, `e4a`, `e4b`, `e5b`, `e6a`, `e6b`, `e6c`).
3. Click **Save Version -> Save & Run All (Commit)** or click **Run All**.
4. Download `validation_results.zip` (~1 MB) from the **Output** tab.


In [ ]:
# 1. Environment and resource check
import os, psutil, shutil, torch
print("CPU cores:", os.cpu_count())
print(f"Total RAM: {psutil.virtual_memory().total / (1024**3):.1f} GB")
print(f"CUDA available: {torch.cuda.is_available()}" + (f" ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else ""))
print(f"Disk free (/kaggle/working): {shutil.disk_usage('/kaggle/working').free / (1024**3):.1f} GB")
print(f"Disk free (/tmp):             {shutil.disk_usage('/tmp').free / (1024**3):.1f} GB")


In [ ]:
# 2. Clone repo, verify preregistration freeze, and install optivision in editable mode
WORK = "/kaggle/working"
REPO = f"{WORK}/optivision-rag"
!rm -rf {REPO} && git clone -q --depth 1 --branch easy-mode https://github.com/Daemon-VI/optivision-rag.git {REPO}
!pip install -q -e "{REPO}[bench]"
!cd {REPO} && git log --oneline -1
!python {REPO}/scripts/easy_validation/freeze.py --verify 1 2 3


In [ ]:
# 3. Symlink data/easy to /tmp so 31 GB of vector shards unpack to /tmp (73 GB space)
#    instead of filling up /kaggle/working (19.5 GB quota)
import os, pathlib, shutil
TMP_DATA = "/tmp/optdata/easy"
os.makedirs(TMP_DATA, exist_ok=True)
repo_data = pathlib.Path(REPO) / "data" / "easy"
repo_data.parent.mkdir(parents=True, exist_ok=True)
if repo_data.is_symlink() or repo_data.exists():
    if repo_data.is_symlink():
        repo_data.unlink()
    else:
        shutil.rmtree(repo_data, ignore_errors=True)
os.symlink(TMP_DATA, repo_data)
print(f"Symlinked {repo_data} -> {TMP_DATA}")


In [ ]:
# 4. Discover and ingest all batch archives (or unpacked results_e* folders) from /kaggle/input
import glob, os, pathlib, shutil, subprocess, sys

zips = sorted(glob.glob("/kaggle/input/**/results_*.zip", recursive=True))
print(f"Found {len(zips)} batch archive(s) in /kaggle/input:")
for z in zips:
    name = os.path.basename(z)
    print(f"Ingesting {name}...")
    cmd = [sys.executable, f"{REPO}/scripts/easy_validation/ingest.py", z]
    res = subprocess.run(cmd, cwd=REPO, env={**os.environ, "PYTHONPATH": f"{REPO}/src:{REPO}/scripts"}, capture_output=True, text=True)
    if res.returncode != 0:
        print(f"  Error ingesting {name}:", res.stderr.strip())
    else:
        lines = res.stdout.strip().splitlines()
        print(f"  OK: {lines[0]} ({lines[-1]})")

# Fallback: if Kaggle automatically unzipped any results_e* directory in /kaggle/input
for d in sorted(glob.glob("/kaggle/input/**/results_e*", recursive=True)):
    p = pathlib.Path(d)
    if p.is_dir():
        batch = p.name[len("results_"):]
        dest = pathlib.Path(TMP_DATA) / batch
        if not dest.exists():
            print(f"Linking unpacked batch folder {p} -> {dest}")
            os.symlink(str(p), str(dest))


In [ ]:
# 5. Run the full validation analysis & decision pipeline (Parts 1, 2, and 3)
import os, pathlib, shutil, subprocess, sys, torch

OUT_RESULTS = f"{WORK}/results"
os.makedirs(OUT_RESULTS, exist_ok=True)

# Seed from the repository's Step 1-6 checkpoint so V1a..V1t do not need to be recomputed
repo_ckpt = pathlib.Path(REPO) / "reports" / "easy" / "results" / "_step_checkpoint.json"
out_ckpt = pathlib.Path(OUT_RESULTS) / "_step_checkpoint.json"
if repo_ckpt.exists() and not out_ckpt.exists():
    shutil.copy2(repo_ckpt, out_ckpt)
    print(f"Seeded checkpoint from {repo_ckpt} ({repo_ckpt.stat().st_size // 1024} KB)")

env = {**os.environ, "PYTHONPATH": f"{REPO}/src:{REPO}/scripts"}
threads = str(os.cpu_count() or 4)
for var in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    env[var] = threads
if torch.cuda.is_available():
    env["OPTIVISION_SCORE_DEVICE"] = "cuda"
    print("Using CUDA device for MaxSim scoring (OPTIVISION_SCORE_DEVICE=cuda)")

cmd = [
    sys.executable, f"{REPO}/scripts/easy_validation/run_all.py",
    "--data", f"{REPO}/data/easy",
    "--out", OUT_RESULTS,
]
print("Scoring predictions (Parts 1-3) and computing decision proposals...")
proc = subprocess.run(cmd, cwd=REPO, env=env)
if proc.returncode != 0:
    raise SystemExit(f"run_all.py failed with returncode {proc.returncode}")
print("Validation scoring completed successfully!")


In [ ]:
# 6. Display SUMMARY.md and package reports into validation_results.zip
import os, pathlib, zipfile

summary_file = pathlib.Path(OUT_RESULTS) / "SUMMARY.md"
if summary_file.exists():
    print(summary_file.read_text(encoding="utf-8"))

zip_path = f"{WORK}/validation_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(pathlib.Path(OUT_RESULTS).glob("*")):
        if p.is_file():
            z.write(p, p.name)
print(f"\n===> Created {zip_path} ({os.path.getsize(zip_path)} bytes)!")
print("Download validation_results.zip from the Output tab.")
